# Qwen3-Omni

In [1]:
import os
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from transformers import Qwen3OmniMoeForConditionalGeneration, Qwen3OmniMoeProcessor
from qwen_omni_utils import process_mm_info



/root/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [3]:
CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "Qwen/Qwen3-Omni-30B-A3B-Instruct"

os.environ["HF_HOME"] = CACHE_DIR

model = Qwen3OmniMoeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="flash_attention_2",
    cache_dir=CACHE_DIR,
)
model.disable_talker()

processor = Qwen3OmniMoeProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Qwen3-Omni model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_interleaved', 'interleaved', 'mrope_section'}
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section', 'interleaved'}
You are attempting to use Flash Attention 2 without specifying a torch dtype. This might lead to unexpected behaviour
Loading checkpoint shards: 100%|██████████| 15/15 [00:08<00:00,  1.81it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Qwen3-Omni model loaded.


In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [5]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="qwen3_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [6]:
VALID_LABELS = {"Dementia", "Control"}
USE_AUDIO_IN_VIDEO = True


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio": str(wav_path)},
                {"type": "text",  "text": USER_PROMPT},
            ],
        },
    ]

    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)
    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=USE_AUDIO_IN_VIDEO,
    )
    inputs = inputs.to(model.device).to(model.dtype)

    text_ids, _ = model.generate(
        **inputs,
        return_audio=False,
        use_audio_in_video=USE_AUDIO_IN_VIDEO,
        max_new_tokens=64,
    )
    output = processor.batch_decode(
        text_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output[0]

In [ ]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [ ]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [12]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

[Pitt-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Denoiser, exists=True


Pitt-Denoiser:   0%|          | 1/551 [00:12<1:58:54, 12.97s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Denoiser:   0%|          | 2/551 [00:21<1:37:04, 10.61s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-Denoiser:   1%|          | 3/551 [00:30<1:29:58,  9.85s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-Denoiser: 100%|██████████| 551/551 [1:22:29<00:00,  8.98s/it]

[Pitt-Denoiser]
  Accuracy:    0.5789
  F1:          0.7198
  Control Acc: 0.0868
  Dementia Acc:0.9644
  Valid: 551/551  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

In [14]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

[Pitt-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-FRCRN_SE, exists=True


Pitt-FRCRN_SE:   0%|          | 1/551 [00:09<1:24:10,  9.18s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-FRCRN_SE:   0%|          | 2/551 [00:18<1:24:07,  9.19s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-FRCRN_SE:   1%|          | 3/551 [00:27<1:23:59,  9.20s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-FRCRN_SE: 100%|██████████| 551/551 [1:23:16<00:00,  9.07s/it]

[Pitt-FRCRN_SE]
  Accuracy:    0.5826
  F1:          0.7181
  Control Acc: 0.1157
  Dementia Acc:0.9482
  Valid: 551/551  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

In [16]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

[Pitt-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer, exists=True


Pitt-MossFormer:   0%|          | 1/551 [00:09<1:23:21,  9.09s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-MossFormer:   0%|          | 2/551 [00:18<1:22:30,  9.02s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-MossFormer:   1%|          | 3/551 [00:27<1:22:46,  9.06s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-MossFormer: 100%|██████████| 551/551 [1:23:14<00:00,  9.06s/it]

[Pitt-MossFormer]
  Accuracy:    0.5771
  F1:          0.7223
  Control Acc: 0.0620
  Dementia Acc:0.9806
  Valid: 551/551  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [18]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

[Pitt-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Resemble, exists=True


Pitt-Resemble:   0%|          | 1/551 [00:09<1:22:58,  9.05s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Resemble:   0%|          | 2/551 [00:18<1:22:33,  9.02s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-Resemble:   1%|          | 3/551 [00:27<1:22:56,  9.08s/it]Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-Resemble: 100%|██████████| 551/551 [1:23:36<00:00,  9.10s/it]

[Pitt-Resemble]
  Accuracy:    0.5808
  F1:          0.7200
  Control Acc: 0.0950
  Dementia Acc:0.9612
  Valid: 551/551  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")